# MCP Tool Discovery

This notebook shows the key idea behind MCP: **the server tells the client what tools are available**, including their names, descriptions, and parameter schemas. The agent (or any client) **discovers** this information at runtime before calling any tool.

## Prerequisites

Start the MCP notes server in a separate terminal:

```bash
uv run python tutorials/mcp_agent/notes_server.py
```

The server should be running on `http://127.0.0.1:9001`.

## Step 1 — Connect to the MCP server

We open an SSE connection and initialize a `ClientSession`. This is the same handshake that an ADK agent performs internally.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from contextlib import AsyncExitStack
from mcp import ClientSession
from mcp.client.sse import sse_client

MCP_SERVER_URL = "http://127.0.0.1:9001/sse"

stack = AsyncExitStack()
streams = await stack.enter_async_context(sse_client(MCP_SERVER_URL))
session = await stack.enter_async_context(ClientSession(*streams))
await session.initialize()

print("Connected to MCP server")

## Step 2 — Discover tools (`list_tools`)

This is **the key step**. Before calling anything, the client asks the server: *"What tools do you have?"*

The server responds with the full contract for each tool:
- **name** — how to call it
- **description** — what it does (the LLM reads this!)
- **inputSchema** — exact parameters and types expected

In [ ]:
import json

tools_result = await session.list_tools()

for tool in tools_result.tools:
    print(f"Name:        {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Schema:      {json.dumps(tool.inputSchema, indent=2)}")
    print()

### What the LLM sees

When an ADK agent connects to this MCP server, the LLM receives **exactly** the output above. From that information alone it decides:
- *Which* tool to call (based on the description)
- *What arguments* to pass (based on the schema)

No tool logic is hardcoded in the agent. The server is the single source of truth.

## Step 3 — Call tools

Now we call each tool manually, just like the agent would after discovery.

### 3a — `create_note`

In [ ]:
result = await session.call_tool(
    "create_note",
    {"title": "shopping list", "content": "Eggs, Milk, Bread"},
)
for block in result.content:
    print(block.text)

### 3b — `list_notes`

In [ ]:
result = await session.call_tool("list_notes", {})
for block in result.content:
    print(block.text)

### 3c — `read_note`

In [ ]:
result = await session.call_tool("read_note", {"title": "shopping list"})
for block in result.content:
    print(block.text)

## Step 4 — The full picture

```
┌──────────────┐                    ┌──────────────┐
│   ADK Agent  │  1. list_tools()   │  MCP Server  │
│   (client)   │ ─────────────────> │              │
│              │ <───────────────── │              │
│              │  names, schemas,   │  @mcp.tool() │
│              │  descriptions      │  @mcp.tool() │
│              │                    │  @mcp.tool() │
│  LLM decides │  2. call_tool()    │              │
│  which tool  │ ─────────────────> │  executes    │
│  + arguments │ <───────────────── │  returns     │
└──────────────┘     result         └──────────────┘
```

- **Step 1 (`list_tools`)** — The client discovers what exists. The server returns the full contract (name + description + JSON schema).
- **Step 2 (`call_tool`)** — The LLM picks the right tool and builds the arguments from the schema. The server executes and returns the result.

The agent has **zero hardcoded knowledge** about the tools. If you add a new tool to the server, restart it, and re-run Step 2 above, it will appear automatically.

## Cleanup

In [ ]:
await stack.aclose()
print("Disconnected from MCP server")